# Customer Analytics: Exploratory Data Analysis and KPIs

**Type:** Original supplementary practice  
**Data:** Reproducible synthetic order records created in this notebook

This notebook demonstrates a repeatable EDA workflow: data quality checks, cleaning, derived metrics, customer and product segmentation, and management-facing KPIs. The data are simulated and must not be interpreted as observations from a real business.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 2400
dates = pd.date_range("2024-01-01", periods=540, freq="D")
regions = ["Dhaka", "Chattogram", "Rajshahi", "Khulna"]
categories = ["Electronics", "Home", "Fashion", "Grocery"]
orders = pd.DataFrame({
    "order_id": np.arange(10001, 10001+n),
    "order_date": rng.choice(dates, n),
    "customer_id": rng.integers(1000, 1650, n),
    "region": rng.choice(regions, n, p=[.45, .25, .18, .12]),
    "category": rng.choice(categories, n),
    "units": rng.integers(1, 6, n),
    "unit_price": rng.lognormal(mean=5.0, sigma=.55, size=n).round(2),
    "discount_rate": rng.choice([0, .05, .10, .15, .20], n, p=[.35,.20,.22,.15,.08])
})
orders["revenue"] = (orders["units"] * orders["unit_price"] * (1-orders["discount_rate"])).round(2)
orders["delivery_days"] = np.clip(rng.normal(3.8, 1.5, n), 1, 12).round(1)
orders.loc[rng.choice(orders.index, 20, replace=False), "delivery_days"] = np.nan

orders.head()

In [ ]:
print("Rows, columns:", orders.shape)
display(orders.info())
display(orders.describe(include="all").T)
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())
print("\nMissing values:")
display(orders.isna().sum().to_frame("missing_count"))

## Data quality and preparation

The order identifier should be unique. Missing delivery durations are retained as missing for summary purposes rather than silently replaced with an invented value.

In [ ]:
df = orders.drop_duplicates(subset="order_id").copy()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.to_period("M").dt.to_timestamp()
df["delivery_days_missing"] = df["delivery_days"].isna()

assert df["order_id"].is_unique
assert (df["units"] > 0).all()
assert df["revenue"].ge(0).all()

print("Prepared rows:", len(df))
print("Customers:", df["customer_id"].nunique())
print("Date range:", df["order_date"].min().date(), "to", df["order_date"].max().date())

## Business KPIs

Revenue is the sum of line-level order revenue. Average order value is calculated at the order level here because every row represents one order in this simulated dataset. In real transaction-line data, aggregate to order first.

In [ ]:
kpis = pd.Series({
    "Orders": df["order_id"].nunique(),
    "Customers": df["customer_id"].nunique(),
    "Revenue": df["revenue"].sum(),
    "Average order value": df["revenue"].mean(),
    "Units sold": df["units"].sum(),
    "Average delivery days": df["delivery_days"].mean(),
    "Repeat-customer share": (df["customer_id"].value_counts().gt(1).mean())
})
display(kpis.to_frame("value").round(2))

monthly = df.groupby("month", as_index=False).agg(
    revenue=("revenue","sum"),
    orders=("order_id","nunique"),
    customers=("customer_id","nunique")
)
region_summary = df.groupby("region", as_index=False).agg(
    revenue=("revenue","sum"),
    orders=("order_id","nunique"),
    average_order_value=("revenue","mean")
).sort_values("revenue", ascending=False)
category_summary = df.groupby("category", as_index=False).agg(
    revenue=("revenue","sum"),
    units=("units","sum")
).sort_values("revenue", ascending=False)

display(region_summary.round(2))
display(category_summary.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(monthly["month"], monthly["revenue"], marker="o", linewidth=1.5)
ax.set(title="Monthly revenue (synthetic data)", xlabel="Month", ylabel="Revenue")
ax.grid(alpha=.25)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(region_summary["region"], region_summary["revenue"])
ax.set(title="Revenue by region", xlabel="Region", ylabel="Revenue")
plt.tight_layout()
plt.show()

## Interpretation prompts

1. Which region contributes the most revenue, and does it also have the most orders?
2. Which category has the highest revenue per unit?
3. Is revenue trending consistently, or are there seasonal fluctuations?
4. What additional data would be required to calculate customer lifetime value?

**Limitations:** All records are synthetic. The repeat-customer share is a simple descriptive indicator, not a cohort-retention metric. No causal conclusions can be drawn.